# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

One row represents one content page.

### Time Window

The dataset summarizes activity over the most recent 90-day window. It also contains separate last-30-day and previous-30-day measures that can be used to compare recent performance.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

- impressions_90d
- avg_position
- content_age_days
- days_since_last_update
- word_count

These fields are known before deciding which page should be reviewed.

### Label / Proxy

- ctr

CTR is used as a proxy for engagement opportunity. Pages with meaningful visibility and comparatively low CTR may be candidates for review.

### Context Fields

- content_id
- content_type
- main_intent
- competition_level
- impression_tier
- position_tier

These fields help describe and group pages but are not necessarily direct model inputs.

### Excluded Fields

- client_id: excluded because it identifies the client and is not needed for the decision.
- clicks_90d: excluded because CTR is calculated from clicks and impressions, so using it could leak information about the proxy.
- trend_direction and trend_pct: excluded because they are derived from recent performance changes and may leak outcome-related information.
- provider_used and model_used: excluded because they describe how content was produced, not the search opportunity decision.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
print("Total rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

Total rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0


In [15]:
print("90-day fields:")
print(
    df[
        [
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "days_with_impressions",
            "days_with_sessions"
        ]
    ].describe()
)

print("\nRecent-window fields:")
print(
    df[
        [
            "impressions_last_30d",
            "impressions_prev_30d",
            "clicks_last_30d",
            "clicks_prev_30d"
        ]
    ].describe()
)

90-day fields:
       impressions_90d    clicks_90d  sessions_90d  days_with_impressions  \
count     30000.000000  30000.000000  30000.000000           30000.000000   
mean       5200.366300     16.097333     37.066633              61.454033   
std       16838.019547     75.076958    107.069131              32.689245   
min           1.000000      0.000000      1.000000               1.000000   
25%          81.000000      0.000000      2.000000              31.000000   
50%         731.000000      1.000000      7.000000              81.000000   
75%        3615.250000      7.000000     27.000000              88.000000   
max      517715.000000   4178.000000   4345.000000              88.000000   

       days_with_sessions  
count        30000.000000  
mean            13.096333  
std             17.307759  
min              1.000000  
25%              2.000000  
50%              6.000000  
75%             16.000000  
max             90.000000  

Recent-window fields:
       impressio

In [16]:
required_fields = [
    "impressions_90d",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "ctr"
]

print("Missing values in required fields:")
print(df[required_fields].isnull().sum())

usable = df.dropna(subset=required_fields)

print("\nTotal rows:", len(df))
print("Usable rows:", len(usable))
print("Rows removed:", len(df) - len(usable))

Missing values in required fields:
impressions_90d              0
avg_position                 0
content_age_days             0
days_since_last_update       0
word_count                7699
ctr                          0
dtype: int64

Total rows: 30000
Usable rows: 22301
Rows removed: 7699


In [17]:
feature_columns = [
    "impressions_90d",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

feature_frame = usable[
    ["content_id"] + feature_columns + ["ctr"]
].copy()

feature_frame.head()

,content_id,impressions_90d,avg_position,content_age_days,days_since_last_update,word_count,ctr
0,content_304f48230142,3803,10.6,187,20,3221.0,0.76
1,content_a1fb4e703a9e,15320,20.3,445,25,2481.0,0.05
2,content_9aa793d4d895,12581,36.5,141,20,3515.0,0.09
4,content_d99b7a2d90ca,19140,44.0,263,14,2803.0,0.13
5,content_d4084a4bc775,3970,8.5,147,20,3080.0,0.03


### Why these five features are available at decision time

- **impressions_90d:** Historical search visibility is already known before choosing a page for review.
- **avg_position:** Historical average search position is available before the decision.
- **content_age_days:** The age of the page is known at the time of review.
- **days_since_last_update:** The last update date is known before prioritization.
- **word_count:** The current page length is known before any recommendation is made.

In [18]:
leaky_frame = feature_frame.copy()

leaky_frame["clicks_90d"] = usable["clicks_90d"]

leaky_frame[
    ["content_id", "impressions_90d", "clicks_90d", "ctr"]
].head()

,content_id,impressions_90d,clicks_90d,ctr
0,content_304f48230142,3803,29,0.76
1,content_a1fb4e703a9e,15320,7,0.05
2,content_9aa793d4d895,12581,11,0.09
4,content_d99b7a2d90ca,19140,24,0.13
5,content_d4084a4bc775,3970,1,0.03


### Leakage Trap

I deliberately added `clicks_90d` to the feature frame.

This is leakage because CTR is calculated from clicks and impressions. Using `clicks_90d` would give the model direct information about the proxy it is meant to estimate or rank.

This could produce unrealistically strong results that would not represent a fair decision-time system.

In [19]:
clean_feature_frame = leaky_frame.drop(columns=["clicks_90d"])

print("Columns after removing leakage:")
print(clean_feature_frame.columns.tolist())

Columns after removing leakage:
['content_id', 'impressions_90d', 'avg_position', 'content_age_days', 'days_since_last_update', 'word_count', 'ctr']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits

This dataset is a 90-day snapshot rather than a complete historical panel, so it cannot show every seasonal or long-term pattern.

CTR is only a proxy for opportunity. Low CTR does not prove that a title, description, or page is poor.

The data cannot establish causation or guarantee that changing a page will increase clicks.

Some fields have missing values, so the usable slice may not represent every page equally.

The data also cannot explain user intent, brand effects, search-result features, or external events that may influence performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.